# Flagship v2 — do raw performance metrics misrepresent project-manager contribution?

**All data in the first part is synthetic** (fixed seed). Every project manager with settled milestones is included.
The seven steps follow the study protocol: raw outcomes → assignment → transparent and primary models → uncertainty →
interpretations before/after → sensitivity and omitted confounders → misleading proxies. Hidden synthetic parameters
are read only at the end, by the evaluation harness, to check the method. The last part repeats the key comparison on
**public real-world project data** when it has been fetched (`python scripts/fetch_public_data.py`).

Ground rules: selection rules use observable data only and are fixed in advance; nobody is designated as superior;
adjusted estimates are associations, not causal effects of a person.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 140)
from meritvector_analytics.synthetic import generate
from meritvector_analytics.pipeline import run_pipeline
from meritvector_analytics.flagship import run_flagship

ds = generate()
result = run_pipeline(ds.events, ds.employees, str(ds.config.as_of), manager_ratings=ds.manager_ratings,
                      promotions=ds.promotions, protected_attributes=ds.protected_attributes, n_boot=100)
f = run_flagship(result, ds.manager_ratings)
steps = f["steps"]
print(f["title"]); print(len(f["_people"]), "project managers")

Do raw performance metrics misrepresent project-manager contribution?
298 project managers


## Step 1 — Raw delivery outcomes

In [2]:
print(steps["1_raw_outcomes"]["plain_language"])
pd.Series(steps["1_raw_outcomes"]["on_time_rate_quantiles"], name="raw on-time rate")

Raw on-time rates range from 0% (10th percentile) to 78% (90th). The measured conditions of each person's projects alone predict 46% of the differences in their raw schedule variance — before anything about the person is considered.


p10    0.000
p25    0.136
p50    0.366
p75    0.583
p90    0.778
Name: raw on-time rate, dtype: float64

## Step 2 — Who gets which conditions

In [3]:
print(steps["2_assignment"]["plain_language"])
pd.DataFrame(steps["2_assignment"]["conditions_by_job_family"])

Assignment is not random: project complexity clusters by project manager (ICC 0.34; Kruskal–Wallis p = 7e-10), and people with harder portfolios have lower raw on-time rates (Spearman -0.39). Raw rates therefore partly measure the portfolio, not the person.


,job_family,mean_complexity,mean_resource_constraint,mean_dependencies
0,program_manager,3.16,0.31,2.71
1,project_manager,2.94,0.32,2.50
2,technical_project_manager,3.47,0.29,2.70


## Step 3 — Transparent baseline (cluster-robust OLS) and the primary within-person model

In [4]:
print(steps["3_models"]["plain_language"])
b = pd.DataFrame(steps["3_models"]["transparent_baseline"]["coefficients"]).set_index("term")
b["within_person"] = pd.Series(steps["3_models"]["primary_model"]["within_person_coefficients"])
b

Both models agree on the direction of every main condition: more complexity, dependencies and staffing shortfall go with more slippage. The within-person model also compares each person with themselves across conditions, which is designed for organisations where stronger people are given harder projects.


,estimate,ci95,p_value,within_person
term,,,,
size_band,-0.004356,"[-1.1859439258985915, 1.1772321534582582]",9.942351e-01,-0.519925
technical_complexity,2.638170,"[1.8545471905878381, 3.4217925604265558]",4.153969e-11,5.183450
dependency_count,2.260398,"[1.8576921819875198, 2.6631039604928772]",3.765532e-28,2.195362
resource_constraint,17.675910,"[14.192091730845664, 21.159728237829153]",2.670348e-23,21.379295
scope_changes_to_date,4.371203,"[3.7946269618553674, 4.947778538236248]",6.073144e-50,4.207686
team_size,0.346003,"[0.03985740962962009, 0.6521480337943473]",2.675087e-02,0.329280


## Step 4 — Uncertainty and unsupported conclusions

In [5]:
print(steps["4_uncertainty"]["plain_language"])
pd.Series(steps["4_uncertainty"]["conclusion_counts"], name="people")

A raw dashboard would single out 59 people as top performers and 60 as weak. Once conditions and uncertainty are accounted for, the evidence supports 24% of the 'top' verdicts and 22% of the 'weak' ones. Typical uncertainty is ±7 pp; 15 people get no conclusion at all.


within_expected_range        195
above_context_expectation     45
below_context_expectation     43
insufficient_evidence         15
Name: people, dtype: int64

## Step 5 — The same records, read before and after contextual analysis

In [6]:
print("Selection rule:", steps["5_interpretations"]["selection_rule"], "\n")
for e in steps["5_interpretations"]["examples"]:
    print(e["person"], "|", e["conditions"])
    print("  raw dashboard:", e["raw_dashboard_reading"])
    print("  MeritVector:  ", e["meritvector_reading"], "\n")

Selection rule: The people whose raw and adjusted positions disagree most (observable data only, fixed in advance). 

PM-243 | complexity 5.0/5, staffing shortfall 0.49, 5.8 dependencies
  raw dashboard: On-time rate 0% — in the bottom fifth; reads as under-performing.
  MeritVector:   Across 13 settled milestones, this person's projects slipped on average 32.2% of the planned interval. A simple comparison with similar projects across the organisation would expect 32.8%. The primary model also compares the person with themselves across easier and harder projects, which corrects for harder work tending to go to particular people. It estimates delivery was better than these conditions predict, by about 8.1 percentage points — and the uncertainty range (+1.2 to +14.9) does not include zero. The raw on-time rate is 0% on projects of average complexity 5.0 of 5. Raw rates mostly reflect project difficulty, which is why the adjusted comparison is shown. 

PM-130 | complexity 1.0/5, staffing 

## Step 6 — Sensitivity to assumptions and omitted confounders

In [7]:
print(steps["6_sensitivity"]["plain_language"])
pd.DataFrame(steps["6_sensitivity"]["alternative_specifications"]).T

Share of strong conclusions that would survive an unmeasured condition as influential as each measured one: technical complexity 51%, resource constraint 45%, dependency count 45% — so about 47% are robust to a confounder of that size and the rest are provisional. Dropping complexity and dependencies from the model changes 31% of conclusions — so conclusions depend on measuring context well.


,spearman_with_base,share_conclusions_changed
no_project_variance_component,0.992012,0.157718
add_business_unit_fixed_effects,0.991633,0.057047
drop_resource_constraint,0.905107,0.187919
drop_complexity_and_dependencies,0.445829,0.305369
winsorised_outcome_5_95,0.994845,0.020134
random_intercept_only_v0_1,0.929175,0.140940
no_context_adjustment,0.271234,0.342282
huber_robust_residuals,0.877087,NaN


## Step 7 — How misleading proxies distort assessment

In [8]:
print(steps["7_proxies"]["plain_language"])
pd.DataFrame(steps["7_proxies"]["proxies"])

No activity or perception proxy tracks the context-adjusted evidence closely (strongest: e-mail activity, Spearman -0.21); meeting activity, e-mail activity even point the wrong way. A productivity tracker built on these signals would reward visibility and circumstance; MeritVector never turns activity into credit.


,proxy,spearman_with_adjusted_evidence,p_value
0,Raw on-time rate,0.276934,2.234265e-06
1,Raw mean schedule variance (lower is better),0.302349,2.152455e-07
2,Meeting activity,-0.207376,4.457276e-04
3,E-mail activity,-0.211197,3.467585e-04
4,Stakeholder perception score,0.065331,2.742245e-01
5,Tasks completed (volume),-0.094228,1.137242e-01
6,Historical manager rating,0.063741,2.852326e-01


## Check against the hidden synthetic truth (evaluation only)

Possible only because the organisation is synthetic. The hidden parameters were not used anywhere above.

In [9]:
from meritvector_analytics.evaluation import flagship_truth_check
t = flagship_truth_check(result, f, ds.ground_truth)
print(f"Raw top fifth truly above average: {t['raw_top_quintile_truly_above_average']:.0%}")
print(f"'Above expectation' conclusions truly above average: {t['adjusted_above_truly_above_average']:.0%}")
print("True effects of the step-5 examples (pp):", {k: round(v, 1) for k, v in t["example_true_effects_pp"].items()})
pd.DataFrame(t["signals"])

Raw top fifth truly above average: 54%
'Above expectation' conclusions truly above average: 100%
True effects of the step-5 examples (pp): {'PM-243': 6.2, 'PM-130': -9.3, 'PM-284': 10.5, 'PM-076': -14.0}


,signal,spearman_with_truth,true_top_quartile_placed_in_bottom_half
0,MeritVector context-adjusted estimate,0.787740,0.093333
1,Raw on-time rate,0.260868,0.400000
2,Raw mean schedule variance (lower is better),0.284421,0.413333
3,Meeting activity,-0.225784,0.560000
4,E-mail activity,-0.235760,0.560000
5,Stakeholder perception score,0.126628,0.413333
6,Tasks completed (volume),-0.133890,0.586667
7,Historical manager rating,0.070437,0.573333


## Does it hold on real data? (public project datasets)

In [10]:
from meritvector_analytics.public_data import sources
if all(s.available for k, s in sources().items() if k in ("ebspm", "desharnais")):
    from meritvector_analytics.public_data.studies import desharnais, ebspm
    e = ebspm(n_boot=100)
    print(e["plain_language"], "\n")
    print(desharnais(n_boot=500)["plain_language"])
    team = e["results"]["log_cost_per_fp"]["team_indicators"]
    display(pd.DataFrame({k: {"raw %": v["raw_pct"], "adjusted %": v["adjusted_pct"]} for k, v in team.items()}).T.round(1))
else:
    print("Public datasets not fetched — run: python scripts/fetch_public_data.py")

Across 488 real projects from 3 organisations, project size and measured context account for 23% of the variation in cost per function point, team indicators for 5% and the organisation itself for 6%. Adjusting for size and context leaves the order of organisations unchanged, but the gaps widen (adjusted gaps are 1.3× the raw ones): raw figures can understate, not only overstate, differences. Projects flagged as having a fixed, experienced team cost 47% less per function point on raw figures, but 9% less once size, context and organisation are accounted for (95% -38% to +32%) — raw team comparisons overstate the team effect. A context model trained on two organisations does not beat a simple median when applied to the third. 



In 77 Canadian projects, each extra year of manager experience is associated with +6.4% hours per function point after adjustment (95% -1.2% to +14.7%). The interval includes zero: years of experience alone do not show a clear link to productivity — a reason not to treat tenure as a proxy for contribution.


,raw %,adjusted %
Fixed_experienced_team,-47.1,-9.5
Many_team_changes_unexperienced_team,95.0,66.7
Steady_heartbeat,-44.9,-18.6


## Reading the result

Raw metrics mostly describe the portfolio a person was given. Adjusting for measured conditions — and saying "no
conclusion" when the evidence is thin — changes most raw verdicts, and the adjusted reading tracks the (synthetic)
truth far better than raw rates, activity, perception scores or historical ratings. On real projects the same
distortion appears for team indicators. None of this validates individual estimates in a real organisation; that
requires a governed pilot.